# Attempt 002: frozen joint preference model — predict

Load the selected QLoRA adapter and base checkpoint locally. Package versions are
restored from wheels produced by training. Internet is disabled. The adapter digest,
preprocessing digest, split assignments, and temperature are fixed before this run.


In [ ]:
import os,sys,json,subprocess,shutil,base64,zlib
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
WORK=Path('/kaggle/working')
parents=[Path('/kaggle/input/llm-preference-qwen3-4b-train-002'),Path('/kaggle/input/notebooks/arseniytrushin/llm-preference-qwen3-4b-train-002')]
TRAIN=next(p for p in parents if (p/'joint_run/adapter/adapter_model.safetensors').exists())
runtime=json.loads((TRAIN/'runtime_requirements.json').read_text())
subprocess.run([sys.executable,'-m','pip','install','-q','--no-index','--no-deps',
    '--find-links',str(TRAIN/'runtime_wheels'),*runtime],check=True)
MODULES=WORK/'bundled_code'
shutil.copytree(TRAIN/'bundled_code',MODULES,dirs_exist_ok=True)
SELECTION={'job': 'qwen4b1024', 'kernel': 'arseniytrushin/llm-preference-qwen3-4b-train-002', 'version_number': 1, 'adapter_sha256': '8afc09d728698c7bcd02e47f9d427166b4b07b3644c848052cc66c9b4f08e39d', 'temperature': 0.7220597450036835, 'config': {'attempt': '002', 'model_id': 'Qwen/Qwen3-4B', 'model_source': 'qwen-lm/qwen-3/transformers/4b/1', 'max_length': 1024, 'seed': 20261102, 'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'learning_rate': 5e-05, 'weight_decay': 0.01, 'epochs': 1, 'train_batch_size': 4, 'eval_batch_size': 4, 'gradient_accumulation_steps': 4, 'group_training_by_length': True, 'eval_steps': 200, 'selection_dev_rows': 1024, 'max_train_seconds': 21600, 'preprocessing_sha256': 'a2e047f0e732dd4b1da19fb18baf49ecda888e7602ba3fb353a39c327a93e672', 'split_ids_sha256': '066f7ab82a3f14464ce0ef67bfd8b8b34ed190679bfea5c1fea80abc3aa4ecd1'}, 'development': {'log_loss': 0.9773837072913087, 'accuracy': 0.5273233553776541, 'balanced_accuracy': 0.5247367963128498, 'macro_f1': 0.5246110105826401, 'brier': 0.5831936008352668}, 'calibrated_development': {'log_loss': 0.9688254519759955, 'accuracy': 0.5273233553776541, 'balanced_accuracy': 0.5247367963128498, 'macro_f1': 0.5246110105826401, 'brier': 0.5776749160955102}, 'epochs_completed': 0.6101930770568794, 'stopped_for_budget': True, 'confirmation_used_for_selection': False}
(WORK/'model_selection.json').write_text(json.dumps(SELECTION,indent=2))


In [ ]:
(MODULES/'arena/joint_infer.py').write_text('"""Frozen adapter evaluation and internet-disabled competition prediction."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom peft import PeftModel\nfrom transformers import AutoTokenizer,TrainingArguments\nfrom .data import CLASSES,TEXT_COLUMNS,parse_turns,json_write\nfrom .joint_data import JointTokenizer,PreferenceCollator,read_competition,split_from_ids\nfrom .joint_train import load_base_model,predict_swapped,metrics,temper,PreferenceTrainer\n\n\ndef load_inference_model(model_path,adapter_path,config):\n    model,_=load_base_model(model_path,config,training=False)\n    return PeftModel.from_pretrained(model,str(adapter_path),is_trainable=False).eval()\n\n\ndef run(raw,model_path,adapter,config,training_summary,out,mode,selection,split_path=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    assert hashlib.sha256((Path(adapter)/\'adapter_model.safetensors\').read_bytes()).hexdigest()==selection[\'adapter_sha256\']\n    assert training_summary[\'temperature\']==selection[\'temperature\']\n    assert not selection[\'confirmation_used_for_selection\']\n    if config.get(\'preprocessing_sha256\'):\n        assert hashlib.sha256(Path(__file__).with_name(\'joint_data.py\').read_bytes()).hexdigest()==config[\'preprocessing_sha256\']\n    tokenizer=AutoTokenizer.from_pretrained(str(adapter),local_files_only=True,padding_side=\'right\')\n    if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token\n    if mode==\'confirm\':\n        if split_path is None:raise ValueError(\'Confirmation requires explicit split IDs\')\n        frame,_=read_competition(raw)\n        parts=split_from_ids(frame,json.loads(Path(split_path).read_text()))\n        selected=frame.iloc[parts[\'confirmation\']]\n        labels=selected[CLASSES].values.argmax(1)\n        counts=frame.iloc[parts[\'train\']][CLASSES].sum().to_numpy(dtype=float)\n        prior=counts/counts.sum()\n    else:\n        selected=pd.read_csv(Path(raw)/\'test.csv\')\n        for name in TEXT_COLUMNS:selected[name]=selected[name].map(parse_turns)\n        labels=None\n    original,swapped,diagnostics=JointTokenizer(tokenizer,config[\'max_length\']).encode_frame(selected)\n    model=load_inference_model(model_path,adapter,config)\n    model.config.pad_token_id=tokenizer.pad_token_id\n    cuda=torch.cuda.is_available()\n    bf16=cuda and torch.cuda.get_device_capability()[0]>=8\n    args=TrainingArguments(output_dir=str(out/\'evaluation_tmp\'),per_device_eval_batch_size=config[\'eval_batch_size\'],\n        fp16=cuda and not bf16,bf16=bf16,report_to=\'none\',remove_unused_columns=False,\n        dataloader_num_workers=0,dataloader_pin_memory=cuda,disable_tqdm=True)\n    trainer=PreferenceTrainer(model=model,args=args,processing_class=tokenizer,data_collator=PreferenceCollator(tokenizer))\n    p=predict_swapped(trainer,original,swapped,labels)\n    calibrated=temper(p,training_summary[\'temperature\'])\n    if trainer.is_world_process_zero():\n        if mode==\'confirm\':\n            result={\'confirmation\':metrics(labels,calibrated),\'uncalibrated_confirmation\':metrics(labels,p),\n                \'training_prior_confirmation\':metrics(labels,np.tile(prior,(len(labels),1))),\n                \'majority_confirmation\':metrics(labels,np.tile(np.eye(3)[prior.argmax()],(len(labels),1))),\n                \'rows\':len(selected),\'temperature\':training_summary[\'temperature\'],\n                \'model_choices_frozen_before_confirmation\':True,\'diagnostics\':diagnostics}\n            json_write(out/\'confirmation.json\',result)\n            np.savez_compressed(out/\'confirmation_predictions.npz\',ids=selected.id.to_numpy(),labels=labels,probabilities=calibrated)\n            print(json.dumps(result,indent=2),flush=True)\n        else:\n            submission=pd.DataFrame(calibrated,columns=CLASSES)\n            submission.insert(0,\'id\',selected.id.to_numpy())\n            assert submission.id.is_unique and np.isfinite(calibrated).all()\n            assert np.all(calibrated>=0) and np.allclose(calibrated.sum(1),1)\n            assert submission.columns.tolist()==pd.read_csv(Path(raw)/\'sample_submission.csv\').columns.tolist()\n            submission.to_csv(out/\'submission.csv\',index=False)\n            json_write(out/\'prediction_summary.json\',{\'rows\':len(submission),\'classes\':CLASSES,\n                \'temperature\':training_summary[\'temperature\'],\'diagnostics\':diagnostics,\'probabilities_valid\':True})\n            print(\'Wrote submission.csv:\',len(submission),\'rows\',flush=True)\n    trainer.accelerator.wait_for_everyone()\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    for name in [\'raw\',\'model\',\'adapter\',\'config\',\'summary\',\'out\',\'selection\']:\n        parser.add_argument(\'--\'+name,type=Path,required=True)\n    parser.add_argument(\'--split\',type=Path)\n    parser.add_argument(\'--mode\',choices=[\'confirm\',\'predict\'],required=True)\n    args=parser.parse_args()\n    run(args.raw,args.model,args.adapter,json.loads(args.config.read_text()),json.loads(args.summary.read_text()),args.out,args.mode,json.loads(args.selection.read_text()),args.split)\n\n\nif __name__==\'__main__\':main()\n')

In [ ]:
import torch
assert torch.cuda.is_available()
RAW=next(p for p in [Path('/kaggle/input/llm-classification-finetuning'),
    Path('/kaggle/input/competitions/llm-classification-finetuning')] if (p/'test.csv').exists())
candidates=[Path('/kaggle/input/qwen-3/transformers/4b/1'),Path('/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1'),TRAIN/'base_model']
MODEL=next(p for p in candidates if (p/'config.json').exists())
environment=os.environ.copy();environment['PYTHONPATH']=str(MODULES);environment['PYTHONUNBUFFERED']='1'
command=[sys.executable,'-m','torch.distributed.run','--standalone',f'--nproc_per_node={torch.cuda.device_count()}',
    '--module','arena.joint_infer','--raw',str(RAW),'--model',str(MODEL),
    '--adapter',str(TRAIN/'joint_run/adapter'),'--config',str(TRAIN/'joint_run/training_config.json'),
    '--summary',str(TRAIN/'joint_run/summary.json'),'--selection',str(WORK/'model_selection.json'),
    '--out',str(WORK),'--mode','predict']
command += []
with (WORK/'inference_console.log').open('w') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=environment)
    for line in process.stdout: print(line,end='',flush=True);log.write(line);log.flush()
    code=process.wait()
if code:raise RuntimeError(f'Inference failed with exit code {code}')
